In [1]:
import sys
!{sys.executable} -m pip install \
    "timm==1.0.11" \
    "open_clip_torch==2.29.0" \
    "albumentations==1.4.21" \
    "opencv-python-headless==4.10.0.84" \
    "pandas==2.2.3" \
    "scikit-learn==1.5.2" \
    "pyarrow==18.0.0" \
    "tqdm==4.67.1"

import torch, timm, open_clip, albumentations, pandas, sklearn, pyarrow, tqdm, cv2
print("\nAll imports OK")
print("torch:", torch.__version__)
print("CUDA :", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

/venv/main/lib/python3.12/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.21). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()



All imports OK
torch: 2.11.0+cu130
CUDA : True | NVIDIA H100 80GB HBM3


In [2]:
# ============================================================
# Cell 1: Imports, config, environment
# ============================================================

# ---- Thread limits (MUST be set before cv2/torch/numpy import) ----
import os
os.environ['OPENCV_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '4'

import sys
import json
import time
import random
import pickle
import warnings
from pathlib import Path
from copy import deepcopy
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast
import timm
import open_clip
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
cv2.setNumThreads(1)  # belt and suspenders

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

# ... rest of cell 1 unchanged ...

# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# ---- Environment check ----
print("=" * 60)
print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"BF16 support: {torch.cuda.is_bf16_supported()}")
print("=" * 60)

DEVICE = torch.device("cuda")

# ---- Paths ----
DATASET_ROOT = Path("/dataset")
CSV_PATH = DATASET_ROOT / "master_v4.csv"

LOO_ROOT = DATASET_ROOT / "loo_sweep_v2"  # NEW folder, fresh start
LOO_CKPT_DIR = LOO_ROOT / "checkpoints"
LOO_LOG_DIR = LOO_ROOT / "logs"
LOO_RESULTS_DIR = LOO_ROOT / "results"
LOO_STATE_FILE = LOO_ROOT / "sweep_state.json"

for d in [LOO_ROOT, LOO_CKPT_DIR, LOO_LOG_DIR, LOO_RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"\nLOO sweep root: {LOO_ROOT}")

# ---- Config ----
@dataclass
class Config:
    image_size: int = 224
    batch_size: int = 96
    num_workers: int = 12   # conservative, reliable
    val_frac: float = 0.20
    
    # Swin-T
    swin_total_epochs: int = 10
    swin_swa_window: int = 3
    swin_backbone_lr: float = 2e-5
    swin_head_lr: float = 1e-3
    swin_warmup_epochs: int = 1
    
    # CLIP ViT-B/16
    clip_total_epochs: int = 10
    clip_swa_window: int = 3
    clip_backbone_lr: float = 1e-5
    clip_head_lr: float = 5e-4
    clip_warmup_epochs: int = 1
    
    label_smoothing: float = 0.05
    grad_clip_norm: float = 1.0
    weight_decay: float = 0.01
    tta_views: int = 4
    amp_dtype: torch.dtype = torch.bfloat16
    seed: int = SEED

CFG = Config()

# ---- Active datasets (real_and_fake_face_clean_v1 dropped) ----
ALL_DATASETS = [
    "deepfake_and_real_clean_v1",
    "140k_clean_v1",
    "artifact_clean_v2",
    "ffpp_c23",
    "celeb_df_v2",
    "sd_generated",
    "adversarial",
]
SINGLE_CLASS_DATASETS = {"sd_generated", "adversarial"}

# ---- Load and filter master CSV ----
MASTER_DF = pd.read_csv(CSV_PATH)
MASTER_DF = MASTER_DF[MASTER_DF['dataset'].isin(ALL_DATASETS)].reset_index(drop=True)
print(f"\nLoaded master CSV: {len(MASTER_DF):,} rows across {MASTER_DF['dataset'].nunique()} datasets")
print(f"\nPer-dataset counts:")
print(MASTER_DF['dataset'].value_counts().to_string())
print(f"\nLabel balance: {dict(MASTER_DF['label'].value_counts())}")

print("\nCell 1 complete.")

/venv/main/lib/python3.12/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.21). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Python      : 3.12.13
PyTorch     : 2.11.0+cu130
CUDA avail  : True
Device      : NVIDIA H100 80GB HBM3
VRAM        : 85.0 GB
BF16 support: True

LOO sweep root: /dataset/loo_sweep_v2

Loaded master CSV: 427,063 rows across 7 datasets

Per-dataset counts:
dataset
deepfake_and_real_clean_v1    142528
140k_clean_v1                 123056
artifact_clean_v2              78420
ffpp_c23                       38309
celeb_df_v2                    24750
sd_generated                   10000
adversarial                    10000

Label balance: {1: np.int64(223553), 0: np.int64(203510)}

Cell 1 complete.


In [3]:
# ============================================================
# Cell 2: Data pipeline
# ============================================================

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def build_train_augs(image_size, mean, std):
    return A.Compose([
        A.SmallestMaxSize(max_size=int(image_size * 1.15), interpolation=cv2.INTER_AREA),
        A.RandomCrop(height=image_size, width=image_size),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05, scale_limit=0.1, rotate_limit=10,
            border_mode=cv2.BORDER_REFLECT_101, p=0.5,
        ),
        A.OneOf([
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05, p=1.0),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=15, p=1.0),
        ], p=0.5),
        A.ImageCompression(quality_lower=60, quality_upper=100, p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.MotionBlur(blur_limit=5, p=1.0),
            A.GaussNoise(var_limit=(5.0, 20.0), p=1.0),
        ], p=0.2),
        A.CoarseDropout(
            max_holes=4, max_height=image_size // 8, max_width=image_size // 8,
            min_holes=1, min_height=image_size // 16, min_width=image_size // 16,
            fill_value=0, p=0.3,
        ),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])


def build_val_augs(image_size, mean, std):
    return A.Compose([
        A.SmallestMaxSize(max_size=int(image_size * 1.15), interpolation=cv2.INTER_AREA),
        A.CenterCrop(height=image_size, width=image_size),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])


def build_tta_augs(image_size, mean, std):
    base_resize = A.SmallestMaxSize(max_size=int(image_size * 1.15), interpolation=cv2.INTER_AREA)
    norm = A.Normalize(mean=mean, std=std)
    to_tensor = ToTensorV2()
    return [
        A.Compose([base_resize, A.CenterCrop(image_size, image_size), norm, to_tensor]),
        A.Compose([base_resize, A.CenterCrop(image_size, image_size),
                   A.HorizontalFlip(p=1.0), norm, to_tensor]),
        A.Compose([A.SmallestMaxSize(max_size=int(image_size * 1.30), interpolation=cv2.INTER_AREA),
                   A.CenterCrop(image_size, image_size), norm, to_tensor]),
        A.Compose([base_resize, A.CenterCrop(image_size, image_size),
                   A.Rotate(limit=(5, 5), p=1.0, border_mode=cv2.BORDER_REFLECT_101),
                   norm, to_tensor]),
    ]


class DeepfakeDataset(Dataset):
    def __init__(self, df, transform, return_index=False):
        self.paths = df['path'].values
        self.labels = df['label'].values.astype(np.float32)
        self.datasets = df['dataset'].values
        self.transform = transform
        self.return_index = return_index
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((CFG.image_size, CFG.image_size, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        out = self.transform(image=img)
        tensor = out['image']
        if self.return_index:
            return tensor, label, idx
        return tensor, label


def build_balanced_sampler(df):
    """Composite weights: balance both real/fake and source dataset."""
    label_counts = df['label'].value_counts().to_dict()
    dataset_counts = df['dataset'].value_counts().to_dict()
    
    label_w = df['label'].map(lambda l: 1.0 / label_counts[l]).values
    dataset_w = df['dataset'].map(lambda d: 1.0 / dataset_counts[d]).values
    
    weights = label_w * dataset_w
    weights = weights / weights.sum()
    
    return WeightedRandomSampler(
        weights=torch.from_numpy(weights).double(),
        num_samples=len(df),
        replacement=True,
    )


def build_holdout_loaders(holdout_dataset, val_frac=0.20, seed=42):
    """
    Build train/val/test loaders for one LOO holdout.
    Conservative settings: no persistent workers, num_workers=12.
    """
    train_pool_df = MASTER_DF[MASTER_DF['dataset'] != holdout_dataset].reset_index(drop=True)
    test_df = MASTER_DF[MASTER_DF['dataset'] == holdout_dataset].reset_index(drop=True)
    
    # Stratified 80/20 split
    rng = np.random.RandomState(seed)
    train_pool_df['_strat'] = train_pool_df['dataset'] + '_' + train_pool_df['label'].astype(str)
    val_indices = []
    train_indices = []
    for stratum, group in train_pool_df.groupby('_strat'):
        idx = group.index.values
        rng.shuffle(idx)
        n_val = int(round(len(idx) * val_frac))
        val_indices.extend(idx[:n_val])
        train_indices.extend(idx[n_val:])
    
    train_df = train_pool_df.loc[train_indices].drop(columns='_strat').reset_index(drop=True)
    val_df = train_pool_df.loc[val_indices].drop(columns='_strat').reset_index(drop=True)
    
    train_aug = build_train_augs(CFG.image_size, IMAGENET_MEAN, IMAGENET_STD)
    val_aug = build_val_augs(CFG.image_size, IMAGENET_MEAN, IMAGENET_STD)
    
    train_ds = DeepfakeDataset(train_df, train_aug)
    val_ds = DeepfakeDataset(val_df, val_aug)
    test_ds = DeepfakeDataset(test_df, val_aug, return_index=True)
    
    train_sampler = build_balanced_sampler(train_df)
    
    common = dict(
        num_workers=CFG.num_workers,
        pin_memory=True,
        persistent_workers=False,  # safer
        prefetch_factor=2,
    )
    
    train_loader = DataLoader(
        train_ds, batch_size=CFG.batch_size, sampler=train_sampler,
        drop_last=True, **common,
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG.batch_size * 2, shuffle=False,
        drop_last=False, **common,
    )
    test_loader = DataLoader(
        test_ds, batch_size=CFG.batch_size * 2, shuffle=False,
        drop_last=False, **common,
    )
    
    return {
        'train_loader': train_loader,
        'val_loader': val_loader,
        'test_loader': test_loader,
        'train_df': train_df,
        'val_df': val_df,
        'test_df': test_df,
    }


print("Cell 2 complete.")

Cell 2 complete.


In [4]:
# ============================================================
# Cell 3: Model factories (Swin-T + CLIP ViT-B/16)
# ============================================================

class TimmBinaryWrapper(nn.Module):
    """Wrap a timm model with num_classes=1 to squeeze trailing dim."""
    def __init__(self, timm_model):
        super().__init__()
        self.model = timm_model
    def forward(self, x):
        return self.model(x).squeeze(-1)


def build_swin_t(pretrained=True):
    base = timm.create_model(
        'swin_tiny_patch4_window7_224',
        pretrained=pretrained,
        num_classes=1,
        drop_rate=0.1,
        drop_path_rate=0.2,
    )
    backbone_params, head_params = [], []
    for name, p in base.named_parameters():
        if 'head' in name:
            head_params.append(p)
        else:
            backbone_params.append(p)
    model = TimmBinaryWrapper(base)
    return model, backbone_params, head_params


class CLIPViTBinary(nn.Module):
    def __init__(self, pretrained=True, dropout=0.2):
        super().__init__()
        clip_model, _, _ = open_clip.create_model_and_transforms(
            'ViT-B-16',
            pretrained='openai' if pretrained else None,
        )
        self.visual = clip_model.visual
        feat_dim = self.visual.output_dim
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        feats = self.visual(x)
        return self.head(feats).squeeze(-1)


def build_clip_vit_b16(pretrained=True):
    model = CLIPViTBinary(pretrained=pretrained)
    backbone_params = list(model.visual.parameters())
    head_params = list(model.head.parameters())
    return model, backbone_params, head_params


def freeze_backbone(params):
    for p in params:
        p.requires_grad = False

def unfreeze_backbone(params):
    for p in params:
        p.requires_grad = True


def build_optimizer_and_scheduler(
    backbone_params, head_params, backbone_lr, head_lr,
    weight_decay, total_epochs, steps_per_epoch, warmup_epochs=1,
):
    optimizer = torch.optim.AdamW(
        [
            {'params': backbone_params, 'lr': backbone_lr, 'name': 'backbone'},
            {'params': head_params, 'lr': head_lr, 'name': 'head'},
        ],
        weight_decay=weight_decay, betas=(0.9, 0.999), eps=1e-8,
    )
    
    total_steps = total_epochs * steps_per_epoch
    warmup_steps = warmup_epochs * steps_per_epoch
    
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        progress = min(1.0, progress)
        return 0.01 + 0.99 * 0.5 * (1 + np.cos(np.pi * progress))
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler


print("Cell 3 complete.")

Cell 3 complete.


In [5]:
# ============================================================
# Cell 4: Training utilities (SWA only, no EMA, no compile)
# ============================================================

class ManualSWA:
    """SWA via running mean of state_dicts."""
    def __init__(self):
        self.n_averaged = 0
        self.averaged_state = None
        self.non_float_buffers = None
    
    @torch.no_grad()
    def update(self, model):
        sd = model.state_dict()
        if self.averaged_state is None:
            self.averaged_state = {k: v.detach().clone().float() for k, v in sd.items()
                                   if v.dtype.is_floating_point}
            self.non_float_buffers = {k: v.detach().clone() for k, v in sd.items()
                                      if not v.dtype.is_floating_point}
            self.n_averaged = 1
        else:
            self.n_averaged += 1
            for k, v_avg in self.averaged_state.items():
                v_new = sd[k].detach().float()
                v_avg.add_((v_new - v_avg) / self.n_averaged)
    
    def get_averaged_state_dict(self):
        result = {}
        for k, v in self.averaged_state.items():
            result[k] = v.clone()
        for k, v in self.non_float_buffers.items():
            result[k] = v.clone()
        return result


class SmoothBCEWithLogits(nn.Module):
    def __init__(self, smoothing=0.05, pos_weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.pos_weight = pos_weight
    
    def forward(self, logits, targets):
        targets = targets.float()
        smooth = targets * (1 - self.smoothing) + (1 - targets) * self.smoothing
        return F.binary_cross_entropy_with_logits(logits, smooth, pos_weight=self.pos_weight)


@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()
    all_logits, all_labels = [], []
    total_loss, n_samples = 0.0, 0
    
    for batch in loader:
        if len(batch) == 3:
            imgs, lbls, _ = batch
        else:
            imgs, lbls = batch
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        
        with autocast(dtype=CFG.amp_dtype):
            logits = model(imgs)
            loss = criterion(logits, lbls)
        
        total_loss += loss.item() * lbls.size(0)
        n_samples += lbls.size(0)
        all_logits.append(logits.float().cpu())
        all_labels.append(lbls.float().cpu())
    
    all_logits = torch.cat(all_logits).numpy()
    all_labels = torch.cat(all_labels).numpy()
    probs = 1 / (1 + np.exp(-all_logits))
    preds = (probs > 0.5).astype(int)
    
    auroc = roc_auc_score(all_labels, probs) if len(np.unique(all_labels)) > 1 else float('nan')
    acc = accuracy_score(all_labels, preds)
    
    return {
        'loss': total_loss / max(n_samples, 1),
        'acc': float(acc),
        'auroc': float(auroc),
        'n_samples': int(n_samples),
    }


def train_one_epoch(model, loader, optimizer, scheduler, criterion, device, grad_clip_norm):
    model.train()
    total_loss, correct, n_samples = 0.0, 0, 0
    
    pbar = tqdm(loader, desc='train', leave=False, ncols=100)
    for imgs, lbls in pbar:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        with autocast(dtype=CFG.amp_dtype):
            logits = model(imgs)
            loss = criterion(logits, lbls)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()
        scheduler.step()
        
        with torch.no_grad():
            preds = (logits > 0).float()
            correct += (preds == lbls).sum().item()
            total_loss += loss.item() * lbls.size(0)
            n_samples += lbls.size(0)
        
        if pbar.n % 100 == 0:
            pbar.set_postfix({
                'loss': f'{total_loss / max(n_samples, 1):.4f}',
                'acc': f'{correct / max(n_samples, 1):.3f}',
            })
    
    return {
        'train_loss': total_loss / max(n_samples, 1),
        'train_acc': correct / max(n_samples, 1),
    }


@torch.no_grad()
def recompute_bn_stats(model, loader, device, n_batches=200):
    """SWA needs BN running stats recomputed. No-op for LayerNorm models."""
    has_bn = any(isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d))
                 for m in model.modules())
    if not has_bn:
        return
    
    model.train()
    for m in model.modules():
        if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            m.reset_running_stats()
            m.momentum = None
    
    for i, batch in enumerate(loader):
        if i >= n_batches:
            break
        imgs = batch[0].to(device, non_blocking=True)
        with autocast(dtype=CFG.amp_dtype):
            _ = model(imgs)
    
    model.eval()


def train_one_model(
    model_name, model_builder, train_loader, val_loader,
    total_epochs, swa_window, backbone_lr, head_lr, warmup_epochs,
    holdout_name, save_dir=None,
):
    """Trains one model with SWA. No EMA, no compile."""
    save_dir = save_dir or (LOO_CKPT_DIR / f"holdout_{holdout_name}")
    save_dir.mkdir(parents=True, exist_ok=True)
    
    log_dir = LOO_LOG_DIR / f"holdout_{holdout_name}"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{model_name}_epoch_log.jsonl"
    
    print(f"\n{'=' * 60}")
    print(f"  TRAINING {model_name.upper()} | holdout={holdout_name}")
    print(f"{'=' * 60}")
    print(f"  Epochs: {total_epochs}  Warmup: {warmup_epochs}  SWA: last {swa_window}")
    print(f"  Backbone LR: {backbone_lr}  Head LR: {head_lr}")
    
    model, backbone_params, head_params = model_builder()
    model = model.to(DEVICE)
    
    optimizer, scheduler = build_optimizer_and_scheduler(
        backbone_params, head_params, backbone_lr, head_lr,
        weight_decay=CFG.weight_decay, total_epochs=total_epochs,
        steps_per_epoch=len(train_loader), warmup_epochs=warmup_epochs,
    )
    
    train_labels = train_loader.dataset.labels
    n_real = int((train_labels == 0).sum())
    n_fake = int((train_labels == 1).sum())
    pos_weight = torch.tensor([n_real / max(n_fake, 1)], device=DEVICE)
    print(f"  Train balance: real={n_real:,}  fake={n_fake:,}  pos_weight={pos_weight.item():.3f}")
    
    criterion = SmoothBCEWithLogits(smoothing=CFG.label_smoothing, pos_weight=pos_weight)
    eval_criterion = nn.BCEWithLogitsLoss()
    
    swa = ManualSWA()
    swa_start_epoch = total_epochs - swa_window
    
    if warmup_epochs > 0:
        freeze_backbone(backbone_params)
    
    best_val_auroc = -1.0
    best_epoch = -1
    history = []
    
    for epoch in range(total_epochs):
        epoch_start = time.time()
        
        if epoch == warmup_epochs and warmup_epochs > 0:
            print(f"  [epoch {epoch}] Unfreezing backbone")
            unfreeze_backbone(backbone_params)
        
        train_metrics = train_one_epoch(
            model, train_loader, optimizer, scheduler, criterion,
            device=DEVICE, grad_clip_norm=CFG.grad_clip_norm,
        )
        
        val_metrics = evaluate_model(model, val_loader, eval_criterion, DEVICE)
        
        in_swa = epoch >= swa_start_epoch
        if in_swa:
            swa.update(model)
        
        epoch_time = time.time() - epoch_start
        gpu_mem = torch.cuda.max_memory_allocated() / 1e9
        torch.cuda.reset_peak_memory_stats()
        
        log_row = {
            'epoch': epoch,
            'train_loss': train_metrics['train_loss'],
            'train_acc': train_metrics['train_acc'],
            'val_loss': val_metrics['loss'],
            'val_acc': val_metrics['acc'],
            'val_auroc': val_metrics['auroc'],
            'lr_backbone': optimizer.param_groups[0]['lr'],
            'lr_head': optimizer.param_groups[1]['lr'],
            'epoch_time_sec': epoch_time,
            'gpu_mem_peak_gb': gpu_mem,
            'in_swa_window': in_swa,
        }
        history.append(log_row)
        with open(log_path, 'a') as f:
            f.write(json.dumps(log_row) + '\n')
        
        print(f"  [epoch {epoch:2d}/{total_epochs-1}] "
              f"train_loss={train_metrics['train_loss']:.4f} "
              f"val_loss={val_metrics['loss']:.4f} "
              f"val_auroc={val_metrics['auroc']:.4f} "
              f"acc={val_metrics['acc']:.3f} "
              f"time={epoch_time:.0f}s mem={gpu_mem:.1f}GB"
              f"{' [SWA]' if in_swa else ''}")
        
        if val_metrics['auroc'] > best_val_auroc:
            best_val_auroc = val_metrics['auroc']
            best_epoch = epoch
            torch.save(model.state_dict(), save_dir / f"{model_name}_best.pt")
    
    torch.save(swa.get_averaged_state_dict(), save_dir / f"{model_name}_swa.pt")
    
    print(f"\n  Evaluating 2 candidates on val set...")
    candidates = {}
    
    base, _, _ = model_builder()
    base = base.to(DEVICE)
    
    base.load_state_dict(torch.load(save_dir / f"{model_name}_best.pt"))
    candidates['best'] = evaluate_model(base, val_loader, eval_criterion, DEVICE)
    
    base.load_state_dict(swa.get_averaged_state_dict())
    recompute_bn_stats(base, train_loader, DEVICE, n_batches=200)
    candidates['swa'] = evaluate_model(base, val_loader, eval_criterion, DEVICE)
    torch.save(base.state_dict(), save_dir / f"{model_name}_swa.pt")
    
    print(f"\n  Candidate val AUROCs:")
    for name, m in candidates.items():
        print(f"    {name:6s}: AUROC={m['auroc']:.4f}  acc={m['acc']:.4f}")
    
    winner = max(candidates.keys(), key=lambda k: candidates[k]['auroc'])
    print(f"  Winner: {winner} (val AUROC={candidates[winner]['auroc']:.4f})")
    
    del model, base, swa, optimizer, scheduler
    torch.cuda.empty_cache()
    
    return {
        'model_name': model_name,
        'holdout_name': holdout_name,
        'best_epoch': best_epoch,
        'history': history,
        'candidates': {k: {'auroc': v['auroc'], 'acc': v['acc']} for k, v in candidates.items()},
        'winner': winner,
        'winner_path': str(save_dir / f"{model_name}_{winner}.pt"),
        'all_paths': {
            'best': str(save_dir / f"{model_name}_best.pt"),
            'swa': str(save_dir / f"{model_name}_swa.pt"),
        },
        'log_path': str(log_path),
    }


print("Cell 4 complete.")

Cell 4 complete.


In [6]:
# ============================================================
# Cell 5: Per-holdout orchestrator (sequential, CLIP+Swin only)
# ============================================================

@torch.no_grad()
def collect_predictions(model, loader, device, return_indices=False):
    model.eval()
    all_probs, all_labels, all_indices = [], [], []
    
    for batch in loader:
        if len(batch) == 3:
            imgs, lbls, idx = batch
            all_indices.append(idx.numpy())
        else:
            imgs, lbls = batch
        imgs = imgs.to(device, non_blocking=True)
        with autocast(dtype=CFG.amp_dtype):
            logits = model(imgs)
        probs = torch.sigmoid(logits.float()).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(lbls.numpy())
    
    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    if return_indices:
        indices = np.concatenate(all_indices) if all_indices else np.arange(len(labels))
        return probs, labels, indices
    return probs, labels


@torch.no_grad()
def tta_predict(model, df, mean, std, device):
    model.eval()
    tta_transforms = build_tta_augs(CFG.image_size, mean, std)
    
    accumulated, final_labels, final_indices = None, None, None
    for transform in tta_transforms:
        ds = DeepfakeDataset(df, transform, return_index=True)
        loader = DataLoader(
            ds, batch_size=CFG.batch_size * 2, shuffle=False,
            num_workers=CFG.num_workers, pin_memory=True,
            persistent_workers=False, prefetch_factor=2,
        )
        probs, labels, indices = collect_predictions(model, loader, device, return_indices=True)
        if accumulated is None:
            accumulated = probs.copy()
            final_labels = labels
            final_indices = indices
        else:
            accumulated += probs
        del ds, loader
    
    accumulated /= len(tta_transforms)
    return accumulated, final_labels, final_indices


def load_trained_model(model_builder, ckpt_path, device):
    model, _, _ = model_builder()
    state = torch.load(ckpt_path, map_location=device)
    state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}
    model.load_state_dict(state)
    model = model.to(device)
    model.eval()
    return model


def compute_holdout_metrics(probs, labels, holdout_name):
    is_single_class = holdout_name in SINGLE_CLASS_DATASETS
    preds = (probs > 0.5).astype(int)
    
    metrics = {
        'holdout_name': holdout_name,
        'n_samples': int(len(labels)),
        'n_real': int((labels == 0).sum()),
        'n_fake': int((labels == 1).sum()),
        'is_single_class': is_single_class,
        'mean_prob': float(probs.mean()),
        'std_prob': float(probs.std()),
    }
    
    if is_single_class:
        metrics['detection_rate'] = float((preds == 1).mean())
        metrics['mean_fake_prob'] = float(probs.mean())
        metrics['auroc'] = None
        metrics['accuracy'] = None
        metrics['f1'] = None
    else:
        metrics['accuracy'] = float(accuracy_score(labels, preds))
        metrics['auroc'] = float(roc_auc_score(labels, probs))
        metrics['f1'] = float(f1_score(labels, preds))
        cm = confusion_matrix(labels, preds)
        metrics['tn'] = int(cm[0, 0])
        metrics['fp'] = int(cm[0, 1])
        metrics['fn'] = int(cm[1, 0])
        metrics['tp'] = int(cm[1, 1])
        metrics['precision'] = float(metrics['tp'] / max(metrics['tp'] + metrics['fp'], 1))
        metrics['recall'] = float(metrics['tp'] / max(metrics['tp'] + metrics['fn'], 1))
    
    return metrics


def run_one_holdout(holdout_name):
    """Sequential CLIP + Swin training for one holdout."""
    holdout_start = time.time()
    
    print(f"\n{'#' * 70}")
    print(f"# HOLDOUT: {holdout_name}")
    print(f"{'#' * 70}")
    
    holdout_ckpt_dir = LOO_CKPT_DIR / f"holdout_{holdout_name}"
    holdout_ckpt_dir.mkdir(parents=True, exist_ok=True)
    holdout_results_dir = LOO_RESULTS_DIR / f"holdout_{holdout_name}"
    holdout_results_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n[1/5] Building dataloaders...")
    t0 = time.time()
    holdout_loaders = build_holdout_loaders(holdout_name)
    print(f"      train={len(holdout_loaders['train_df']):,} "
          f"val={len(holdout_loaders['val_df']):,} "
          f"test={len(holdout_loaders['test_df']):,}  "
          f"({time.time() - t0:.1f}s)")
    
    # ---- Train Swin-T ----
    print(f"\n[2/5] Training Swin-T...")
    swin_result = train_one_model(
        model_name='swin_t', model_builder=build_swin_t,
        train_loader=holdout_loaders['train_loader'],
        val_loader=holdout_loaders['val_loader'],
        total_epochs=CFG.swin_total_epochs, swa_window=CFG.swin_swa_window,
        backbone_lr=CFG.swin_backbone_lr, head_lr=CFG.swin_head_lr,
        warmup_epochs=CFG.swin_warmup_epochs, holdout_name=holdout_name,
    )
    torch.cuda.empty_cache()
    
    # ---- Train CLIP ViT ----
    print(f"\n      Training CLIP ViT-B/16...")
    clip_result = train_one_model(
        model_name='clip_vit', model_builder=build_clip_vit_b16,
        train_loader=holdout_loaders['train_loader'],
        val_loader=holdout_loaders['val_loader'],
        total_epochs=CFG.clip_total_epochs, swa_window=CFG.clip_swa_window,
        backbone_lr=CFG.clip_backbone_lr, head_lr=CFG.clip_head_lr,
        warmup_epochs=CFG.clip_warmup_epochs, holdout_name=holdout_name,
    )
    torch.cuda.empty_cache()
    
    # ---- Meta-learner ----
    print(f"\n[3/5] Collecting val predictions for meta-learner...")
    
    model_results = {'swin_t': swin_result, 'clip_vit': clip_result}
    model_builders = {'swin_t': build_swin_t, 'clip_vit': build_clip_vit_b16}
    val_probs_per_model = {}
    val_labels_ref = None
    
    for mname, mres in model_results.items():
        winner_path = mres['winner_path']
        print(f"      Loading {mname} ({mres['winner']})")
        m = load_trained_model(model_builders[mname], winner_path, DEVICE)
        probs, labels = collect_predictions(m, holdout_loaders['val_loader'], DEVICE)
        val_probs_per_model[mname] = probs
        if val_labels_ref is None:
            val_labels_ref = labels
        del m
        torch.cuda.empty_cache()
    
    val_features = np.stack([
        val_probs_per_model['swin_t'],
        val_probs_per_model['clip_vit'],
    ], axis=1)
    
    print(f"      Fitting meta-learner on {len(val_labels_ref):,} val samples...")
    meta_learner = LogisticRegression(
        C=1.0, max_iter=1000, class_weight='balanced',
        solver='lbfgs', random_state=CFG.seed,
    )
    meta_learner.fit(val_features, val_labels_ref)
    
    coefs = meta_learner.coef_[0]
    print(f"      Meta-learner coefficients:")
    print(f"        Swin-T  : {coefs[0]:+.3f}")
    print(f"        CLIP ViT: {coefs[1]:+.3f}")
    print(f"        intercept: {meta_learner.intercept_[0]:+.3f}")
    
    val_ensemble = meta_learner.predict_proba(val_features)[:, 1]
    val_ensemble_auroc = roc_auc_score(val_labels_ref, val_ensemble)
    print(f"      Meta-learner val AUROC: {val_ensemble_auroc:.4f}")
    
    with open(holdout_ckpt_dir / "meta_learner.pkl", 'wb') as f:
        pickle.dump(meta_learner, f)
    
    # ---- TTA test eval ----
    print(f"\n[4/5] TTA inference on test set ({len(holdout_loaders['test_df']):,} images)...")
    
    test_df = holdout_loaders['test_df']
    test_probs_per_model = {}
    test_labels_ref = None
    test_indices_ref = None
    
    for mname, mres in model_results.items():
        print(f"      TTA: {mname}")
        m = load_trained_model(model_builders[mname], mres['winner_path'], DEVICE)
        probs, labels, indices = tta_predict(m, test_df, IMAGENET_MEAN, IMAGENET_STD, DEVICE)
        test_probs_per_model[mname] = probs
        if test_labels_ref is None:
            test_labels_ref = labels
            test_indices_ref = indices
        del m
        torch.cuda.empty_cache()
    
    test_features = np.stack([
        test_probs_per_model['swin_t'],
        test_probs_per_model['clip_vit'],
    ], axis=1)
    test_ensemble_probs = meta_learner.predict_proba(test_features)[:, 1]
    
    # ---- Metrics ----
    print(f"\n[5/5] Computing metrics...")
    
    per_model_metrics = {}
    for mname, probs in test_probs_per_model.items():
        per_model_metrics[mname] = compute_holdout_metrics(probs, test_labels_ref, holdout_name)
    ensemble_metrics = compute_holdout_metrics(test_ensemble_probs, test_labels_ref, holdout_name)
    
    print(f"\n      RESULTS for '{holdout_name}':")
    if holdout_name in SINGLE_CLASS_DATASETS:
        print(f"        (Single-class — detection rate)")
        for mname, m in per_model_metrics.items():
            print(f"        {mname:10s}: detection_rate={m['detection_rate']:.4f}")
        print(f"        ENSEMBLE  : detection_rate={ensemble_metrics['detection_rate']:.4f}")
    else:
        for mname, m in per_model_metrics.items():
            print(f"        {mname:10s}: AUROC={m['auroc']:.4f}  acc={m['accuracy']:.4f}  f1={m['f1']:.4f}")
        print(f"        ENSEMBLE  : AUROC={ensemble_metrics['auroc']:.4f}  "
              f"acc={ensemble_metrics['accuracy']:.4f}  f1={ensemble_metrics['f1']:.4f}")
    
    per_image_df = pd.DataFrame({
        'idx': test_indices_ref,
        'path': [test_df['path'].iloc[i] for i in test_indices_ref],
        'true_label': test_labels_ref.astype(int),
        'swin_t_prob': test_probs_per_model['swin_t'],
        'clip_vit_prob': test_probs_per_model['clip_vit'],
        'ensemble_prob': test_ensemble_probs,
    })
    per_image_df.to_parquet(holdout_results_dir / "per_image_predictions.parquet", index=False)
    
    holdout_total_time = time.time() - holdout_start
    
    summary = {
        'holdout_name': holdout_name,
        'total_time_sec': holdout_total_time,
        'total_time_min': holdout_total_time / 60,
        'train_size': int(len(holdout_loaders['train_df'])),
        'val_size': int(len(holdout_loaders['val_df'])),
        'test_size': int(len(holdout_loaders['test_df'])),
        'per_model_metrics': per_model_metrics,
        'ensemble_metrics': ensemble_metrics,
        'meta_learner': {
            'coefficients': coefs.tolist(),
            'intercept': float(meta_learner.intercept_[0]),
            'val_auroc': float(val_ensemble_auroc),
        },
        'model_winners': {
            'swin_t': swin_result['winner'],
            'clip_vit': clip_result['winner'],
        },
    }
    
    with open(holdout_results_dir / "summary.json", 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"\n      Holdout '{holdout_name}' complete in {holdout_total_time / 60:.1f} min")
    print(f"{'#' * 70}\n")
    
    return summary


print("Cell 5 complete. Ready to run holdouts.")

Cell 5 complete. Ready to run holdouts.


In [7]:
# ============================================================
# Cell 6: Run celeb_df_v2 as validation holdout
# ============================================================

celeb_summary = run_one_holdout('celeb_df_v2')

print("\n" + "=" * 70)
print("CELEB-DF V2 VALIDATION HOLDOUT COMPLETE")
print("=" * 70)
print(f"Total time: {celeb_summary['total_time_min']:.1f} min")
print(f"\nKey metrics:")
print(f"  Swin-T   test AUROC: {celeb_summary['per_model_metrics']['swin_t']['auroc']:.4f}")
print(f"  CLIP ViT test AUROC: {celeb_summary['per_model_metrics']['clip_vit']['auroc']:.4f}")
print(f"  Ensemble test AUROC: {celeb_summary['ensemble_metrics']['auroc']:.4f}")
print(f"\nDecision:")
print(f"  > 0.90: launch full sweep")
print(f"  0.80-0.90: reasonable, decide case by case")
print(f"  < 0.80: investigate before continuing")


######################################################################
# HOLDOUT: celeb_df_v2
######################################################################

[1/5] Building dataloaders...
      train=321,849 val=80,464 test=24,750  (0.5s)

[2/5] Training Swin-T...

  TRAINING SWIN_T | holdout=celeb_df_v2
  Epochs: 10  Warmup: 1  SWA: last 3
  Backbone LR: 2e-05  Head LR: 0.001


  Train balance: real=152,907  fake=168,942  pos_weight=0.905


train:   0%|                                                               | 0/3352 [00:00<?, ?it/s]

Exception in thread Thread-7 (_pin_memory_loop):
Traceback (most recent call last):
  File "/venv/main/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/venv/main/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/venv/main/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/venv/main/lib/python3.12/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/venv/main/lib/python3.12/site-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/venv/main/lib/python3.12/mu

KeyboardInterrupt: 

In [7]:
# ============================================================
# Cell 7: Suppress harmless DataLoader cleanup spam
# ============================================================
import sys
import warnings

warnings.filterwarnings("ignore")

class _NoIgnoredExceptions:
    """Filter out the harmless 'Exception ignored in __del__' DataLoader spam."""
    def __init__(self, stream):
        self.stream = stream
        self._buffer = ""
    
    def write(self, s):
        if any(marker in s for marker in [
            "Exception ignored",
            "_MultiProcessingDataLoaderIter",
            "_shutdown_workers",
            "is_alive",
            "AssertionError: can only test a child process",
            "can only test a child process",
        ]):
            return
        self.stream.write(s)
    
    def flush(self):
        self.stream.flush()
    
    def __getattr__(self, name):
        return getattr(self.stream, name)

if not isinstance(sys.stderr, _NoIgnoredExceptions):
    sys.stderr = _NoIgnoredExceptions(sys.stderr)

print("Stderr filter installed. DataLoader cleanup spam will be suppressed.")

Stderr filter installed. DataLoader cleanup spam will be suppressed.


In [8]:
# ============================================================
# Cell 8: Full LOO sweep (resumable)
# ============================================================

# Order holdouts from easiest to hardest so we get the cheap wins early
# and have more confidence as we go. celeb_df_v2 is already done.
SWEEP_ORDER = [
    "ffpp_c23",                    # Faces, similar to FF++ training data — should be easiest
    "140k_clean_v1",               # Real/StyleGAN — model has seen similar
    "artifact_clean_v2",           # Mixed artifacts — model has seen similar
    "deepfake_and_real_clean_v1",  # Standard deepfakes — moderate
    "sd_generated",                # Single-class — different metric
    "adversarial",                 # Single-class — different metric
]

def load_sweep_state():
    if LOO_STATE_FILE.exists():
        with open(LOO_STATE_FILE) as f:
            return json.load(f)
    return {"completed": [], "results": {}}

def save_sweep_state(state):
    with open(LOO_STATE_FILE, 'w') as f:
        json.dump(state, f, indent=2, default=str)


# ---- Pre-populate state with already-completed celeb_df_v2 ----
state = load_sweep_state()

celeb_summary_path = LOO_RESULTS_DIR / "holdout_celeb_df_v2" / "summary.json"
if celeb_summary_path.exists() and "celeb_df_v2" not in state["completed"]:
    with open(celeb_summary_path) as f:
        celeb_summary = json.load(f)
    state["completed"].append("celeb_df_v2")
    state["results"]["celeb_df_v2"] = celeb_summary
    save_sweep_state(state)
    print(f"Imported celeb_df_v2 from previous run.")

print(f"\nSweep state at start:")
print(f"  Completed: {state['completed']}")
print(f"  Remaining: {[h for h in SWEEP_ORDER if h not in state['completed']]}")
print()

# ---- Run sweep ----
sweep_start = time.time()

for holdout in SWEEP_ORDER:
    if holdout in state["completed"]:
        print(f"\n>>> SKIPPING {holdout} (already complete)")
        continue
    
    print(f"\n{'>' * 70}")
    print(f">>> STARTING HOLDOUT {len(state['completed']) + 1}/{len(SWEEP_ORDER) + 1}: {holdout}")
    print(f">>> Sweep elapsed so far: {(time.time() - sweep_start) / 60:.1f} min")
    print(f"{'>' * 70}")
    
    try:
        summary = run_one_holdout(holdout)
        state["completed"].append(holdout)
        state["results"][holdout] = summary
        save_sweep_state(state)
        print(f">>> Completed {holdout}, state saved.")
    except Exception as e:
        print(f"\n!!! ERROR on {holdout}: {type(e).__name__}: {e}")
        print(f"!!! State saved with {len(state['completed'])} holdouts completed.")
        print(f"!!! Re-run this cell to resume from {holdout}.")
        save_sweep_state(state)
        raise

sweep_total = time.time() - sweep_start
print(f"\n{'=' * 70}")
print(f"SWEEP COMPLETE")
print(f"{'=' * 70}")
print(f"Total time: {sweep_total / 60:.1f} min ({sweep_total / 3600:.2f} hours)")
print(f"Total holdouts: {len(state['completed'])}")


# ============================================================
# Final aggregate report
# ============================================================
print(f"\n\n{'=' * 70}")
print(f"FINAL LEAVE-ONE-OUT CROSS-DATASET RESULTS")
print(f"{'=' * 70}")

print(f"\n{'Holdout':<32}{'Swin AUROC':<14}{'CLIP AUROC':<14}{'Ensemble':<12}")
print("-" * 72)

mixed_aurocs = []
single_class_rates = []

for holdout in SWEEP_ORDER + ["celeb_df_v2"]:
    if holdout not in state["results"]:
        continue
    summary = state["results"][holdout]
    
    is_single = holdout in SINGLE_CLASS_DATASETS
    
    if is_single:
        swin_m = summary['per_model_metrics']['swin_t']
        clip_m = summary['per_model_metrics']['clip_vit']
        ens_m = summary['ensemble_metrics']
        print(f"{holdout:<32}"
              f"{swin_m['detection_rate']:.4f} (det)  "
              f"{clip_m['detection_rate']:.4f} (det)  "
              f"{ens_m['detection_rate']:.4f} (det)")
        single_class_rates.append((holdout, ens_m['detection_rate']))
    else:
        swin_m = summary['per_model_metrics']['swin_t']
        clip_m = summary['per_model_metrics']['clip_vit']
        ens_m = summary['ensemble_metrics']
        print(f"{holdout:<32}{swin_m['auroc']:<14.4f}{clip_m['auroc']:<14.4f}{ens_m['auroc']:<12.4f}")
        mixed_aurocs.append((holdout, ens_m['auroc']))

if mixed_aurocs:
    avg_auroc = np.mean([a for _, a in mixed_aurocs])
    print(f"\nMean ensemble AUROC across {len(mixed_aurocs)} mixed-class holdouts: {avg_auroc:.4f}")

if single_class_rates:
    avg_det = np.mean([d for _, d in single_class_rates])
    print(f"Mean ensemble detection rate across {len(single_class_rates)} single-class holdouts: {avg_det:.4f}")

print(f"\nResults saved to: {LOO_STATE_FILE}")
print(f"Per-holdout details: {LOO_RESULTS_DIR}")


Sweep state at start:
  Completed: ['celeb_df_v2', 'ffpp_c23', '140k_clean_v1', 'artifact_clean_v2', 'deepfake_and_real_clean_v1']
  Remaining: ['sd_generated', 'adversarial']


>>> SKIPPING ffpp_c23 (already complete)

>>> SKIPPING 140k_clean_v1 (already complete)

>>> SKIPPING artifact_clean_v2 (already complete)

>>> SKIPPING deepfake_and_real_clean_v1 (already complete)

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
>>> STARTING HOLDOUT 6/7: sd_generated
>>> Sweep elapsed so far: 0.0 min
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

######################################################################
# HOLDOUT: sd_generated
######################################################################

[1/5] Building dataloaders...
      train=333,649 val=83,414 test=10,000  (0.5s)

[2/5] Training Swin-T...

  TRAINING SWIN_T | holdout=sd_generated
  Epochs: 10  Warmup: 1  SWA: last 3
  Backbone LR: 2e-05  Head LR: 0.001


  Train balance: real=162,807  fake=170,842  pos_weight=0.953


train:   0%|                                                               | 0/3475 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [9]:
#!/usr/bin/env python3
"""
Standalone script to run remaining LOO holdouts: sd_generated and adversarial.
Runs outside Jupyter, with all DataLoader cleanup spam redirected to /dev/null.
"""

# ============================================================
# CRITICAL: Thread caps MUST come before any heavy imports
# ============================================================
import os
os.environ['OPENCV_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'
os.environ['NUMEXPR_NUM_THREADS'] = '4'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ============================================================
# Redirect stderr to /dev/null to kill AssertionError spam
# ============================================================
import sys

class DevNull:
    def write(self, *args, **kwargs): pass
    def flush(self, *args, **kwargs): pass
    def __getattr__(self, name): return lambda *a, **k: None

# Save original stderr for real errors we want to see
_real_stderr = sys.stderr

class FilteredStderr:
    """Pass through real errors, drop DataLoader cleanup spam."""
    def __init__(self, real):
        self.real = real
        self._buffer = ""
    def write(self, s):
        # Drop anything that smells like DataLoader cleanup noise
        if any(marker in s for marker in [
            "Exception ignored",
            "_MultiProcessingDataLoaderIter",
            "_shutdown_workers",
            "is_alive",
            "can only test a child process",
            "AssertionError",
            "Traceback (most recent call last)",
            "dataloader.py",
            "__del__",
            "^",
        ]):
            return
        self.real.write(s)
    def flush(self):
        self.real.flush()
    def __getattr__(self, name):
        return getattr(self.real, name)

sys.stderr = FilteredStderr(_real_stderr)

# Also suppress warnings module
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# Now the real imports
# ============================================================
import json
import time
import random
import pickle
from pathlib import Path
from copy import deepcopy
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast

import timm
import open_clip
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
cv2.setNumThreads(1)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

# ============================================================
# Reproducibility + device
# ============================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda")
print(f"Device: {torch.cuda.get_device_name(0)}  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB", flush=True)

# ============================================================
# Paths and config
# ============================================================
DATASET_ROOT = Path("/dataset")
CSV_PATH = DATASET_ROOT / "master_v4.csv"
LOO_ROOT = DATASET_ROOT / "loo_sweep_v2"
LOO_CKPT_DIR = LOO_ROOT / "checkpoints"
LOO_LOG_DIR = LOO_ROOT / "logs"
LOO_RESULTS_DIR = LOO_ROOT / "results"
LOO_STATE_FILE = LOO_ROOT / "sweep_state.json"

for d in [LOO_ROOT, LOO_CKPT_DIR, LOO_LOG_DIR, LOO_RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

@dataclass
class Config:
    image_size: int = 224
    batch_size: int = 96
    num_workers: int = 12
    val_frac: float = 0.20
    swin_total_epochs: int = 10
    swin_swa_window: int = 3
    swin_backbone_lr: float = 2e-5
    swin_head_lr: float = 1e-3
    swin_warmup_epochs: int = 1
    clip_total_epochs: int = 10
    clip_swa_window: int = 3
    clip_backbone_lr: float = 1e-5
    clip_head_lr: float = 5e-4
    clip_warmup_epochs: int = 1
    label_smoothing: float = 0.05
    grad_clip_norm: float = 1.0
    weight_decay: float = 0.01
    tta_views: int = 4
    amp_dtype: torch.dtype = torch.bfloat16
    seed: int = SEED

CFG = Config()

ALL_DATASETS = [
    "deepfake_and_real_clean_v1",
    "140k_clean_v1",
    "artifact_clean_v2",
    "ffpp_c23",
    "celeb_df_v2",
    "sd_generated",
    "adversarial",
]
SINGLE_CLASS_DATASETS = {"sd_generated", "adversarial"}

MASTER_DF = pd.read_csv(CSV_PATH)
MASTER_DF = MASTER_DF[MASTER_DF['dataset'].isin(ALL_DATASETS)].reset_index(drop=True)
print(f"Loaded master CSV: {len(MASTER_DF):,} rows", flush=True)

# ============================================================
# Dataset + augmentations
# ============================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def build_train_augs():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=15, val_shift_limit=10, p=1.0),
        ], p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.GaussNoise(var_limit=(5.0, 20.0), p=1.0),
            A.ImageCompression(quality_lower=70, quality_upper=95, p=1.0),
        ], p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.08, rotate_limit=8, border_mode=cv2.BORDER_REFLECT_101, p=0.5),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def build_val_augs():
    return A.Compose([
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def build_tta_augs():
    return [
        A.Compose([A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]),
        A.Compose([A.HorizontalFlip(p=1.0), A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]),
        A.Compose([A.CenterCrop(height=200, width=200, p=1.0), A.Resize(224, 224), A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]),
        A.Compose([A.Rotate(limit=(5, 5), p=1.0, border_mode=cv2.BORDER_REFLECT_101), A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2()]),
    ]

class DeepfakeDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.paths = self.df['path'].values
        self.labels = self.df['label'].values.astype(np.float32)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(image=img)['image']
        return img, self.labels[idx]

def build_balanced_sampler(df):
    label_counts = df['label'].value_counts().to_dict()
    dataset_counts = df['dataset'].value_counts().to_dict()
    weights = np.zeros(len(df), dtype=np.float64)
    for i, row in enumerate(df.itertuples(index=False)):
        lw = 1.0 / label_counts[row.label]
        dw = 1.0 / dataset_counts[row.dataset]
        weights[i] = lw * dw
    weights = weights / weights.sum() * len(df)
    return WeightedRandomSampler(weights=torch.DoubleTensor(weights), num_samples=len(df), replacement=True)

def build_holdout_loaders(holdout_name):
    train_val_df = MASTER_DF[MASTER_DF['dataset'] != holdout_name].reset_index(drop=True)
    test_df = MASTER_DF[MASTER_DF['dataset'] == holdout_name].reset_index(drop=True)
    train_val_df = train_val_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    val_n = int(len(train_val_df) * CFG.val_frac)
    val_df = train_val_df.iloc[:val_n].reset_index(drop=True)
    train_df = train_val_df.iloc[val_n:].reset_index(drop=True)
    print(f"      train={len(train_df):,} val={len(val_df):,} test={len(test_df):,}", flush=True)
    train_ds = DeepfakeDataset(train_df, build_train_augs())
    val_ds = DeepfakeDataset(val_df, build_val_augs())
    test_ds = DeepfakeDataset(test_df, build_val_augs())
    sampler = build_balanced_sampler(train_df)
    train_loader = DataLoader(
        train_ds, batch_size=CFG.batch_size, sampler=sampler,
        num_workers=CFG.num_workers, pin_memory=True,
        persistent_workers=False, prefetch_factor=2, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG.batch_size, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True,
        persistent_workers=False, prefetch_factor=2,
    )
    test_loader = DataLoader(
        test_ds, batch_size=CFG.batch_size, shuffle=False,
        num_workers=CFG.num_workers, pin_memory=True,
        persistent_workers=False, prefetch_factor=2,
    )
    return {
        'train_loader': train_loader, 'val_loader': val_loader, 'test_loader': test_loader,
        'train_df': train_df, 'val_df': val_df, 'test_df': test_df,
    }

# ============================================================
# Models
# ============================================================
class TimmBinaryWrapper(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
    def forward(self, x):
        out = self.backbone(x)
        if out.ndim == 2 and out.shape[1] == 1:
            return out.squeeze(1)
        return out

def build_swin_t():
    m = timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, num_classes=1)
    return TimmBinaryWrapper(m)

class CLIPViTBinary(nn.Module):
    def __init__(self):
        super().__init__()
        model, _, _ = open_clip.create_model_and_transforms('ViT-B-16', pretrained='openai')
        self.visual = model.visual
        feat_dim = self.visual.output_dim
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(512, 1),
        )
    def forward(self, x):
        feat = self.visual(x)
        out = self.head(feat).squeeze(1)
        return out

def build_clip_vit_b16():
    return CLIPViTBinary()

def freeze_backbone(backbone_params):
    for p in backbone_params:
        p.requires_grad = False

def unfreeze_backbone(backbone_params):
    for p in backbone_params:
        p.requires_grad = True

def build_optimizer_and_scheduler(model, is_clip, total_steps, warmup_steps, backbone_lr, head_lr, wd):
    if is_clip:
        backbone_params = list(model.visual.parameters())
        head_params = list(model.head.parameters())
    else:
        backbone_params = [p for n, p in model.backbone.named_parameters() if 'head' not in n]
        head_params = [p for n, p in model.backbone.named_parameters() if 'head' in n]
    optimizer = torch.optim.AdamW([
        {'params': backbone_params, 'lr': backbone_lr},
        {'params': head_params, 'lr': head_lr},
    ], weight_decay=wd)
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + np.cos(np.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler, backbone_params

# ============================================================
# Training utilities
# ============================================================
class ManualSWA:
    def __init__(self):
        self.state = None
        self.n = 0
    def update(self, model):
        sd = {k: v.detach().clone().float() for k, v in model.state_dict().items()}
        sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
        if self.state is None:
            self.state = sd
            self.n = 1
        else:
            for k in self.state:
                self.state[k] = (self.state[k] * self.n + sd[k]) / (self.n + 1)
            self.n += 1
    def apply_to(self, model):
        sd = {k: v.to(next(model.parameters()).dtype) for k, v in self.state.items()}
        model.load_state_dict(sd, strict=False)

class SmoothBCEWithLogits(nn.Module):
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing
    def forward(self, logits, targets):
        targets = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy_with_logits(logits, targets)

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device, grad_clip_norm):
    model.train()
    total_loss, correct, n_samples = 0.0, 0, 0
    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(dtype=CFG.amp_dtype):
            logits = model(imgs)
            loss = criterion(logits, lbls)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()
        scheduler.step()
        with torch.no_grad():
            preds = (torch.sigmoid(logits.float()) > 0.5).float()
            correct += (preds == lbls).sum().item()
            n_samples += lbls.size(0)
            total_loss += loss.item() * lbls.size(0)
    return {'loss': total_loss / n_samples, 'acc': correct / n_samples}

@torch.no_grad()
def evaluate_model(model, loader, criterion, device):
    model.eval()
    all_logits, all_labels = [], []
    total_loss = 0.0
    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        with autocast(dtype=CFG.amp_dtype):
            logits = model(imgs)
            loss = criterion(logits, lbls)
        total_loss += loss.item() * lbls.size(0)
        all_logits.append(logits.float().cpu())
        all_labels.append(lbls.float().cpu())
    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy()
    probs = 1.0 / (1.0 + np.exp(-logits))
    acc = ((probs > 0.5) == labels).mean()
    try:
        auroc = roc_auc_score(labels, probs)
    except ValueError:
        auroc = float('nan')
    return {'loss': total_loss / len(labels), 'acc': float(acc), 'auroc': float(auroc), 'probs': probs, 'labels': labels}

def train_one_model(model_name, model_builder, train_loader, val_loader,
                    total_epochs, swa_window, backbone_lr, head_lr, warmup_epochs, holdout_name):
    save_dir = LOO_CKPT_DIR / f"holdout_{holdout_name}"
    save_dir.mkdir(parents=True, exist_ok=True)
    log_dir = LOO_LOG_DIR / f"holdout_{holdout_name}"
    log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{model_name}_epoch_log.jsonl"
    print(f"\n{'=' * 60}", flush=True)
    print(f"  TRAINING {model_name.upper()} | holdout={holdout_name}", flush=True)
    print(f"{'=' * 60}", flush=True)
    print(f"  Epochs: {total_epochs}  Warmup: {warmup_epochs}  SWA: last {swa_window}", flush=True)
    print(f"  Backbone LR: {backbone_lr}  Head LR: {head_lr}", flush=True)

    train_df_for_pw = train_loader.dataset.df
    real_n = (train_df_for_pw['label'] == 0).sum()
    fake_n = (train_df_for_pw['label'] == 1).sum()
    pos_weight = real_n / max(1, fake_n)
    print(f"  Train balance: real={real_n:,}  fake={fake_n:,}  pos_weight={pos_weight:.3f}", flush=True)

    model = model_builder().to(DEVICE)
    is_clip = 'clip' in model_name
    steps_per_epoch = len(train_loader)
    total_steps = steps_per_epoch * total_epochs
    warmup_steps = steps_per_epoch * warmup_epochs
    optimizer, scheduler, backbone_params = build_optimizer_and_scheduler(
        model, is_clip, total_steps, warmup_steps, backbone_lr, head_lr, CFG.weight_decay,
    )
    criterion = SmoothBCEWithLogits(smoothing=CFG.label_smoothing)
    eval_criterion = nn.BCEWithLogitsLoss()
    swa = ManualSWA()
    swa_start_epoch = total_epochs - swa_window
    best_auroc = -1.0
    best_epoch = -1
    best_path = save_dir / f"{model_name}_best.pt"
    swa_path = save_dir / f"{model_name}_swa.pt"

    freeze_backbone(backbone_params)

    for epoch in range(total_epochs):
        t0 = time.time()
        if epoch == warmup_epochs:
            print(f"  [epoch {epoch}] Unfreezing backbone", flush=True)
            unfreeze_backbone(backbone_params)
        train_metrics = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, DEVICE, CFG.grad_clip_norm)
        val_metrics = evaluate_model(model, val_loader, eval_criterion, DEVICE)
        in_swa = epoch >= swa_start_epoch
        if in_swa:
            swa.update(model)
        if val_metrics['auroc'] > best_auroc:
            best_auroc = val_metrics['auroc']
            best_epoch = epoch
            torch.save(model.state_dict(), best_path)
        mem_gb = torch.cuda.max_memory_allocated() / 1e9
        torch.cuda.reset_peak_memory_stats()
        elapsed = time.time() - t0
        marker = " [SWA]" if in_swa else ""
        print(f"  [epoch {epoch:2d}/{total_epochs - 1}] train_loss={train_metrics['loss']:.4f} "
              f"val_loss={val_metrics['loss']:.4f} val_auroc={val_metrics['auroc']:.4f} "
              f"acc={val_metrics['acc']:.3f} time={elapsed:.0f}s mem={mem_gb:.1f}GB{marker}", flush=True)
        with open(log_path, 'a') as f:
            f.write(json.dumps({
                'epoch': epoch, 'train_loss': train_metrics['loss'], 'train_acc': train_metrics['acc'],
                'val_loss': val_metrics['loss'], 'val_acc': val_metrics['acc'], 'val_auroc': val_metrics['auroc'],
                'time_s': elapsed, 'mem_gb': mem_gb, 'in_swa': in_swa,
            }) + '\n')

    if swa.state is not None:
        torch.save(swa.state, swa_path)

    print(f"\n  Evaluating 2 candidates on val set...", flush=True)
    candidates = {}
    # best
    best_model = model_builder().to(DEVICE)
    best_model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    best_metrics = evaluate_model(best_model, val_loader, eval_criterion, DEVICE)
    candidates['best'] = best_metrics['auroc']
    del best_model
    # swa
    if swa.state is not None:
        swa_model = model_builder().to(DEVICE)
        swa_sd = {k: v.to(next(swa_model.parameters()).dtype) for k, v in swa.state.items()}
        swa_model.load_state_dict(swa_sd, strict=False)
        swa_metrics = evaluate_model(swa_model, val_loader, eval_criterion, DEVICE)
        candidates['swa'] = swa_metrics['auroc']
        del swa_model

    print(f"  Candidate val AUROCs:", flush=True)
    for k, v in candidates.items():
        print(f"    {k:6s}: AUROC={v:.4f}", flush=True)
    winner = max(candidates, key=candidates.get)
    print(f"  Winner: {winner} (val AUROC={candidates[winner]:.4f})", flush=True)

    del model
    torch.cuda.empty_cache()

    return {
        'winner': winner,
        'best_epoch': best_epoch,
        'candidates': candidates,
        'best_path': str(best_path),
        'swa_path': str(swa_path) if swa.state is not None else None,
    }

def load_trained_model(model_builder, path):
    model = model_builder().to(DEVICE)
    sd = torch.load(path, map_location=DEVICE)
    sd = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
    # Cast SWA float32 states back to model dtype
    target_dtype = next(model.parameters()).dtype
    sd = {k: (v.to(target_dtype) if v.dtype.is_floating_point else v) for k, v in sd.items()}
    model.load_state_dict(sd, strict=False)
    model.eval()
    return model

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    for imgs, lbls in loader:
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(dtype=CFG.amp_dtype):
            logits = model(imgs)
        probs = torch.sigmoid(logits.float()).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(lbls.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)

@torch.no_grad()
def tta_predict(model, df, tta_transforms):
    all_tta_probs = []
    for tta in tta_transforms:
        ds = DeepfakeDataset(df, tta)
        loader = DataLoader(ds, batch_size=CFG.batch_size, shuffle=False,
                            num_workers=CFG.num_workers, pin_memory=True,
                            persistent_workers=False, prefetch_factor=2)
        probs, labels = collect_predictions(model, loader)
        all_tta_probs.append(probs)
    mean_probs = np.mean(all_tta_probs, axis=0)
    return mean_probs, labels

def compute_holdout_metrics(labels, probs, holdout_name):
    is_single = holdout_name in SINGLE_CLASS_DATASETS
    metrics = {}
    preds = (probs > 0.5).astype(int)
    if is_single:
        metrics['detection_rate'] = float((preds == 1).mean())
        metrics['acc'] = float((preds == labels).mean())
    else:
        try:
            metrics['auroc'] = float(roc_auc_score(labels, probs))
        except ValueError:
            metrics['auroc'] = float('nan')
        metrics['acc'] = float(accuracy_score(labels, preds))
        metrics['f1'] = float(f1_score(labels, preds))
    return metrics

def run_one_holdout(holdout_name):
    holdout_dir = LOO_RESULTS_DIR / f"holdout_{holdout_name}"
    holdout_dir.mkdir(parents=True, exist_ok=True)
    t_start = time.time()
    print(f"\n{'#' * 70}", flush=True)
    print(f"# HOLDOUT: {holdout_name}", flush=True)
    print(f"{'#' * 70}", flush=True)

    print(f"\n[1/5] Building dataloaders...", flush=True)
    loaders = build_holdout_loaders(holdout_name)

    print(f"\n[2/5] Training Swin-T...", flush=True)
    swin_result = train_one_model(
        'swin_t', build_swin_t,
        loaders['train_loader'], loaders['val_loader'],
        CFG.swin_total_epochs, CFG.swin_swa_window,
        CFG.swin_backbone_lr, CFG.swin_head_lr, CFG.swin_warmup_epochs, holdout_name,
    )
    torch.cuda.empty_cache()

    print(f"\n      Training CLIP ViT-B/16...", flush=True)
    clip_result = train_one_model(
        'clip_vit', build_clip_vit_b16,
        loaders['train_loader'], loaders['val_loader'],
        CFG.clip_total_epochs, CFG.clip_swa_window,
        CFG.clip_backbone_lr, CFG.clip_head_lr, CFG.clip_warmup_epochs, holdout_name,
    )
    torch.cuda.empty_cache()

    print(f"\n[3/5] Collecting val predictions for meta-learner...", flush=True)
    swin_path = swin_result['best_path'] if swin_result['winner'] == 'best' else swin_result['swa_path']
    clip_path = clip_result['best_path'] if clip_result['winner'] == 'best' else clip_result['swa_path']
    print(f"      Loading swin_t ({swin_result['winner']})", flush=True)
    swin_model = load_trained_model(build_swin_t, swin_path)
    print(f"      Loading clip_vit ({clip_result['winner']})", flush=True)
    clip_model = load_trained_model(build_clip_vit_b16, clip_path)

    swin_val_probs, val_labels = collect_predictions(swin_model, loaders['val_loader'])
    clip_val_probs, _ = collect_predictions(clip_model, loaders['val_loader'])
    X_val = np.stack([swin_val_probs, clip_val_probs], axis=1)
    print(f"      Fitting meta-learner on {len(val_labels):,} val samples...", flush=True)
    meta = LogisticRegression(max_iter=1000, C=1.0)
    meta.fit(X_val, val_labels)
    meta_val_probs = meta.predict_proba(X_val)[:, 1]
    meta_val_auroc = roc_auc_score(val_labels, meta_val_probs)
    print(f"      Meta-learner coefficients:", flush=True)
    print(f"        Swin-T  : {meta.coef_[0][0]:+.3f}", flush=True)
    print(f"        CLIP ViT: {meta.coef_[0][1]:+.3f}", flush=True)
    print(f"        intercept: {meta.intercept_[0]:+.3f}", flush=True)
    print(f"      Meta-learner val AUROC: {meta_val_auroc:.4f}", flush=True)

    print(f"\n[4/5] TTA inference on test set ({len(loaders['test_df']):,} images)...", flush=True)
    tta_transforms = build_tta_augs()
    print(f"      TTA: swin_t", flush=True)
    swin_test_probs, test_labels = tta_predict(swin_model, loaders['test_df'], tta_transforms)
    print(f"      TTA: clip_vit", flush=True)
    clip_test_probs, _ = tta_predict(clip_model, loaders['test_df'], tta_transforms)
    X_test = np.stack([swin_test_probs, clip_test_probs], axis=1)
    ens_test_probs = meta.predict_proba(X_test)[:, 1]

    del swin_model, clip_model
    torch.cuda.empty_cache()

    print(f"\n[5/5] Computing metrics...", flush=True)
    swin_m = compute_holdout_metrics(test_labels, swin_test_probs, holdout_name)
    clip_m = compute_holdout_metrics(test_labels, clip_test_probs, holdout_name)
    ens_m = compute_holdout_metrics(test_labels, ens_test_probs, holdout_name)

    print(f"\n      RESULTS for '{holdout_name}':", flush=True)
    is_single = holdout_name in SINGLE_CLASS_DATASETS
    if is_single:
        print(f"        swin_t    : detection_rate={swin_m['detection_rate']:.4f}  acc={swin_m['acc']:.4f}", flush=True)
        print(f"        clip_vit  : detection_rate={clip_m['detection_rate']:.4f}  acc={clip_m['acc']:.4f}", flush=True)
        print(f"        ENSEMBLE  : detection_rate={ens_m['detection_rate']:.4f}  acc={ens_m['acc']:.4f}", flush=True)
    else:
        print(f"        swin_t    : AUROC={swin_m['auroc']:.4f}  acc={swin_m['acc']:.4f}  f1={swin_m['f1']:.4f}", flush=True)
        print(f"        clip_vit  : AUROC={clip_m['auroc']:.4f}  acc={clip_m['acc']:.4f}  f1={clip_m['f1']:.4f}", flush=True)
        print(f"        ENSEMBLE  : AUROC={ens_m['auroc']:.4f}  acc={ens_m['acc']:.4f}  f1={ens_m['f1']:.4f}", flush=True)

    # Save per-image predictions
    pred_df = pd.DataFrame({
        'path': loaders['test_df']['path'].values,
        'label': test_labels,
        'swin_prob': swin_test_probs,
        'clip_prob': clip_test_probs,
        'ensemble_prob': ens_test_probs,
    })
    pred_df.to_parquet(holdout_dir / "per_image_predictions.parquet", index=False)

    elapsed_min = (time.time() - t_start) / 60
    summary = {
        'holdout': holdout_name,
        'swin_result': swin_result,
        'clip_result': clip_result,
        'meta_coefs': {'swin': float(meta.coef_[0][0]), 'clip': float(meta.coef_[0][1]), 'intercept': float(meta.intercept_[0])},
        'meta_val_auroc': float(meta_val_auroc),
        'per_model_metrics': {'swin_t': swin_m, 'clip_vit': clip_m},
        'ensemble_metrics': ens_m,
        'elapsed_min': elapsed_min,
    }
    with open(holdout_dir / "summary.json", 'w') as f:
        json.dump(summary, f, indent=2, default=str)
    print(f"\n      Holdout '{holdout_name}' complete in {elapsed_min:.1f} min", flush=True)
    print(f"{'#' * 70}", flush=True)
    return summary

def load_sweep_state():
    if LOO_STATE_FILE.exists():
        with open(LOO_STATE_FILE) as f:
            return json.load(f)
    return {"completed": [], "results": {}}

def save_sweep_state(state):
    with open(LOO_STATE_FILE, 'w') as f:
        json.dump(state, f, indent=2, default=str)

# ============================================================
# Main: run only remaining holdouts
# ============================================================
REMAINING = ["sd_generated", "adversarial"]

def main():
    state = load_sweep_state()
    print(f"\nSweep state at start:", flush=True)
    print(f"  Completed: {state['completed']}", flush=True)
    print(f"  Remaining to run: {[h for h in REMAINING if h not in state['completed']]}", flush=True)

    sweep_start = time.time()
    for holdout in REMAINING:
        if holdout in state["completed"]:
            print(f"\n>>> SKIPPING {holdout} (already complete)", flush=True)
            continue
        print(f"\n{'>' * 70}", flush=True)
        print(f">>> STARTING HOLDOUT: {holdout}", flush=True)
        print(f">>> Sweep elapsed so far: {(time.time() - sweep_start) / 60:.1f} min", flush=True)
        print(f"{'>' * 70}", flush=True)
        try:
            summary = run_one_holdout(holdout)
            state["completed"].append(holdout)
            state["results"][holdout] = summary
            save_sweep_state(state)
            print(f">>> Completed {holdout}, state saved.", flush=True)
        except Exception as e:
            print(f"\n!!! ERROR on {holdout}: {type(e).__name__}: {e}", flush=True)
            save_sweep_state(state)
            raise

    sweep_total = time.time() - sweep_start
    print(f"\n{'=' * 70}", flush=True)
    print(f"SWEEP COMPLETE", flush=True)
    print(f"{'=' * 70}", flush=True)
    print(f"Total time: {sweep_total / 60:.1f} min", flush=True)

    # Final aggregate report
    print(f"\n\n{'=' * 70}", flush=True)
    print(f"FINAL LEAVE-ONE-OUT CROSS-DATASET RESULTS", flush=True)
    print(f"{'=' * 70}", flush=True)
    ORDER = ["celeb_df_v2", "ffpp_c23", "140k_clean_v1", "artifact_clean_v2",
             "deepfake_and_real_clean_v1", "sd_generated", "adversarial"]
    print(f"\n{'Holdout':<32}{'Swin':<18}{'CLIP':<18}{'Ensemble':<18}", flush=True)
    print("-" * 86, flush=True)
    mixed_aurocs = []
    single_class_rates = []
    for h in ORDER:
        if h not in state["results"]:
            continue
        summ = state["results"][h]
        swin_m = summ['per_model_metrics']['swin_t']
        clip_m = summ['per_model_metrics']['clip_vit']
        ens_m = summ['ensemble_metrics']
        is_single = h in SINGLE_CLASS_DATASETS
        if is_single:
            print(f"{h:<32}{swin_m['detection_rate']:.4f} (det)    "
                  f"{clip_m['detection_rate']:.4f} (det)    "
                  f"{ens_m['detection_rate']:.4f} (det)", flush=True)
            single_class_rates.append((h, ens_m['detection_rate']))
        else:
            print(f"{h:<32}{swin_m['auroc']:<18.4f}{clip_m['auroc']:<18.4f}{ens_m['auroc']:<18.4f}", flush=True)
            mixed_aurocs.append((h, ens_m['auroc']))
    if mixed_aurocs:
        avg = np.mean([a for _, a in mixed_aurocs])
        print(f"\nMean ensemble AUROC across {len(mixed_aurocs)} mixed-class holdouts: {avg:.4f}", flush=True)
    if single_class_rates:
        avg_det = np.mean([d for _, d in single_class_rates])
        print(f"Mean ensemble detection rate across {len(single_class_rates)} single-class holdouts: {avg_det:.4f}", flush=True)
    print(f"\nResults saved to: {LOO_STATE_FILE}", flush=True)

if __name__ == "__main__":
    main()

Device: NVIDIA H100 80GB HBM3  VRAM: 85.0 GB
Loaded master CSV: 427,063 rows

Sweep state at start:
  Completed: ['celeb_df_v2', 'ffpp_c23', '140k_clean_v1', 'artifact_clean_v2', 'deepfake_and_real_clean_v1']
  Remaining to run: ['sd_generated', 'adversarial']

>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
>>> STARTING HOLDOUT: sd_generated
>>> Sweep elapsed so far: 0.0 min
>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>

######################################################################
# HOLDOUT: sd_generated
######################################################################

[1/5] Building dataloaders...
      train=333,651 val=83,412 test=10,000

[2/5] Training Swin-T...

  TRAINING SWIN_T | holdout=sd_generated
  Epochs: 10  Warmup: 1  SWA: last 3
  Backbone LR: 2e-05  Head LR: 0.001
  Train balance: real=162,944  fake=170,707  pos_weight=0.955
  [epoch  0/9] train_loss=0.6167 val_loss=0.6083 val_auroc=0.7353 acc=0.662 time=1

In [11]:

import pandas as pd
df = pd.read_parquet('/dataset/loo_sweep_v2/results/holdout_sd_generated/per_image_predictions.parquet')
print(f"Rows: {len(df)}")
print(f"Label values: {df['label'].value_counts().to_dict()}")
print(f"\nSwin prob stats:")
print(df['swin_prob'].describe())
print(f"\nCLIP prob stats:")
print(df['clip_prob'].describe())
print(f"\nEnsemble prob stats:")
print(df['ensemble_prob'].describe())
print(f"\nFraction > 0.5: swin={(df['swin_prob']>0.5).mean():.4f}  clip={(df['clip_prob']>0.5).mean():.4f}  ens={(df['ensemble_prob']>0.5).mean():.4f}")
print(f"Fraction > 0.1: swin={(df['swin_prob']>0.1).mean():.4f}  clip={(df['clip_prob']>0.1).mean():.4f}  ens={(df['ensemble_prob']>0.1).mean():.4f}")
print(f"Fraction > 0.01: swin={(df['swin_prob']>0.01).mean():.4f}  clip={(df['clip_prob']>0.01).mean():.4f}  ens={(df['ensemble_prob']>0.01).mean():.4f}")


Rows: 10000
Label values: {1.0: 10000}

Swin prob stats:
count    10000.000000
mean         0.038857
std          0.032706
min          0.006826
25%          0.026761
50%          0.031044
75%          0.038700
max          0.677712
Name: swin_prob, dtype: float64

CLIP prob stats:
count    10000.000000
mean         0.025987
std          0.017426
min          0.022803
25%          0.024800
50%          0.025275
75%          0.025761
max          0.950429
Name: clip_prob, dtype: float64

Ensemble prob stats:
count    10000.000000
mean         0.006530
std          0.012494
min          0.005258
25%          0.005792
50%          0.005917
75%          0.006142
max          0.867626
Name: ensemble_prob, dtype: float64

Fraction > 0.5: swin=0.0005  clip=0.0004  ens=0.0002
Fraction > 0.1: swin=0.0316  clip=0.0012  ens=0.0008
Fraction > 0.01: swin=0.9996  clip=1.0000  ens=0.0185


In [12]:
#!/usr/bin/env python3
"""
Option 3: Frozen CLIP linear probe for sd_generated holdout.

Replicates the Ojha et al. (CVPR 2023) finding that frozen CLIP features
generalize across generator paradigms while fine-tuned CLIP does not.

Method:
  1. Extract frozen CLIP ViT-B/16 features for the 6 training datasets
  2. Fit logistic regression on top
  3. Evaluate on sd_generated test set (10k images, all fake)
  4. Report detection rate at threshold 0.5 AND at val-optimal threshold

This stays within LOO methodology: sd_generated images are NEVER seen during
feature extraction or LR fitting.
"""

import os
os.environ['OPENCV_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '4'
os.environ['MKL_NUM_THREADS'] = '4'

import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast

import open_clip
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
cv2.setNumThreads(1)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

# ============================================================
# Paths and config
# ============================================================
DATASET_ROOT = Path("/dataset")
CSV_PATH = DATASET_ROOT / "master_v4.csv"
LOO_ROOT = DATASET_ROOT / "loo_sweep_v2"
RESULTS_DIR = LOO_ROOT / "results" / "holdout_sd_generated_frozen_clip"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda")
BATCH_SIZE = 256  # Bigger because no backprop
NUM_WORKERS = 12
SEED = 42

ALL_DATASETS = [
    "deepfake_and_real_clean_v1",
    "140k_clean_v1",
    "artifact_clean_v2",
    "ffpp_c23",
    "celeb_df_v2",
    "sd_generated",
    "adversarial",
]
HOLDOUT = "sd_generated"

# ============================================================
# Load data
# ============================================================
print(f"Device: {torch.cuda.get_device_name(0)}", flush=True)
print(f"Loading master CSV...", flush=True)
df = pd.read_csv(CSV_PATH)
df = df[df['dataset'].isin(ALL_DATASETS)].reset_index(drop=True)

train_val_df = df[df['dataset'] != HOLDOUT].reset_index(drop=True)
test_df = df[df['dataset'] == HOLDOUT].reset_index(drop=True)
print(f"  Train+val: {len(train_val_df):,}", flush=True)
print(f"  Test (sd_generated): {len(test_df):,}", flush=True)
print(f"  Train+val labels: {train_val_df['label'].value_counts().to_dict()}", flush=True)
print(f"  Test labels: {test_df['label'].value_counts().to_dict()}", flush=True)

# Stratified val split (same 80/20 as the main sweep, same seed)
train_val_df = train_val_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
val_n = int(len(train_val_df) * 0.20)
val_df = train_val_df.iloc[:val_n].reset_index(drop=True)
train_df = train_val_df.iloc[val_n:].reset_index(drop=True)
print(f"\n  Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}", flush=True)

# ============================================================
# Frozen CLIP feature extractor
# ============================================================
print(f"\nLoading frozen CLIP ViT-B/16 (OpenAI weights)...", flush=True)
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-16', pretrained='openai'
)
clip_model = clip_model.to(DEVICE).eval()

# Freeze everything
for p in clip_model.parameters():
    p.requires_grad = False

# Get the feature dimension
FEAT_DIM = clip_model.visual.output_dim
print(f"  CLIP feature dim: {FEAT_DIM}", flush=True)

# ============================================================
# Dataset using CLIP's own preprocessing pipeline
# ============================================================
# CRITICAL: use CLIP's expected normalization, not ImageNet's
# OpenAI CLIP uses mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711]
CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD = [0.26862954, 0.26130258, 0.27577711]

def build_clip_transform():
    return A.Compose([
        A.Resize(224, 224),
        A.Normalize(mean=CLIP_MEAN, std=CLIP_STD),
        ToTensorV2(),
    ])

class FeatureDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.paths = self.df['path'].values
        self.labels = self.df['label'].values.astype(np.float32)
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.transform(image=img)['image']
        return img, self.labels[idx]

# ============================================================
# Feature extraction
# ============================================================
@torch.no_grad()
def extract_features(df, name):
    print(f"\n  Extracting features: {name} ({len(df):,} images)", flush=True)
    transform = build_clip_transform()
    ds = FeatureDataset(df, transform)
    loader = DataLoader(
        ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=False, prefetch_factor=2,
    )
    all_feats, all_labels = [], []
    t0 = time.time()
    for i, (imgs, lbls) in enumerate(loader):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast(dtype=torch.bfloat16):
            feats = clip_model.encode_image(imgs)
        feats = feats.float().cpu().numpy()
        all_feats.append(feats)
        all_labels.append(lbls.numpy())
        if (i + 1) % 50 == 0:
            elapsed = time.time() - t0
            done = (i + 1) * BATCH_SIZE
            rate = done / elapsed
            eta = (len(df) - done) / rate
            print(f"    [{done:,}/{len(df):,}] {rate:.0f} img/s  ETA: {eta:.0f}s", flush=True)
    feats = np.concatenate(all_feats, axis=0)
    labels = np.concatenate(all_labels, axis=0)
    elapsed = time.time() - t0
    print(f"    Done in {elapsed:.0f}s. Feature shape: {feats.shape}", flush=True)
    return feats, labels

print(f"\n{'=' * 60}", flush=True)
print(f"FEATURE EXTRACTION", flush=True)
print(f"{'=' * 60}", flush=True)

t_total = time.time()
train_feats, train_labels = extract_features(train_df, "TRAIN")
val_feats, val_labels = extract_features(val_df, "VAL")
test_feats, test_labels = extract_features(test_df, "TEST (sd_generated)")
print(f"\nTotal feature extraction time: {(time.time() - t_total) / 60:.1f} min", flush=True)

# Save features to disk for reuse
np.savez(
    RESULTS_DIR / "frozen_clip_features.npz",
    train_feats=train_feats, train_labels=train_labels,
    val_feats=val_feats, val_labels=val_labels,
    test_feats=test_feats, test_labels=test_labels,
)
print(f"  Features saved to {RESULTS_DIR / 'frozen_clip_features.npz'}", flush=True)

# ============================================================
# L2-normalize features (Ojha et al. recommend this)
# ============================================================
def l2_normalize(x):
    return x / (np.linalg.norm(x, axis=1, keepdims=True) + 1e-8)

train_feats = l2_normalize(train_feats)
val_feats = l2_normalize(val_feats)
test_feats = l2_normalize(test_feats)
print(f"\n  L2-normalized features.", flush=True)

# ============================================================
# Train logistic regression
# ============================================================
print(f"\n{'=' * 60}", flush=True)
print(f"FITTING LOGISTIC REGRESSION", flush=True)
print(f"{'=' * 60}", flush=True)

# Try a few C values; pick best on val AUROC
best_C, best_val_auroc, best_lr = None, -1.0, None
for C in [0.01, 0.1, 1.0, 10.0]:
    print(f"\n  C={C}...", flush=True)
    t0 = time.time()
    lr = LogisticRegression(max_iter=2000, C=C, n_jobs=-1, solver='lbfgs')
    lr.fit(train_feats, train_labels)
    val_probs = lr.predict_proba(val_feats)[:, 1]
    val_auroc = roc_auc_score(val_labels, val_probs)
    print(f"    Fit in {time.time() - t0:.0f}s. Val AUROC: {val_auroc:.4f}", flush=True)
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        best_C = C
        best_lr = lr

print(f"\n  Best C: {best_C}  Val AUROC: {best_val_auroc:.4f}", flush=True)

# ============================================================
# Evaluate on sd_generated test set
# ============================================================
print(f"\n{'=' * 60}", flush=True)
print(f"EVALUATION ON SD_GENERATED TEST SET", flush=True)
print(f"{'=' * 60}", flush=True)

val_probs = best_lr.predict_proba(val_feats)[:, 1]
test_probs = best_lr.predict_proba(test_feats)[:, 1]

# Test set is single-class (all label=1)
print(f"\n  Test prob stats:", flush=True)
print(f"    min:    {test_probs.min():.4f}", flush=True)
print(f"    25%:    {np.percentile(test_probs, 25):.4f}", flush=True)
print(f"    median: {np.median(test_probs):.4f}", flush=True)
print(f"    mean:   {test_probs.mean():.4f}", flush=True)
print(f"    75%:    {np.percentile(test_probs, 75):.4f}", flush=True)
print(f"    max:    {test_probs.max():.4f}", flush=True)

# Detection rate at threshold 0.5 (default)
det_rate_05 = float((test_probs > 0.5).mean())
print(f"\n  Detection rate @ threshold 0.5: {det_rate_05:.4f}  ({int((test_probs > 0.5).sum())}/{len(test_probs)})", flush=True)

# Detection rate at val-optimal threshold (Youden's J statistic)
fpr, tpr, thresholds = roc_curve(val_labels, val_probs)
youden_idx = np.argmax(tpr - fpr)
val_optimal_thresh = float(thresholds[youden_idx])
print(f"\n  Val-optimal threshold (Youden's J): {val_optimal_thresh:.4f}", flush=True)
print(f"    At this threshold, val TPR={tpr[youden_idx]:.4f}, val FPR={fpr[youden_idx]:.4f}", flush=True)

det_rate_youden = float((test_probs > val_optimal_thresh).mean())
print(f"  Detection rate @ val-optimal threshold: {det_rate_youden:.4f}  ({int((test_probs > val_optimal_thresh).sum())}/{len(test_probs)})", flush=True)

# Also report at a few other thresholds for full picture
print(f"\n  Detection rate sweep:", flush=True)
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    rate = float((test_probs > t).mean())
    print(f"    @ {t:.1f}: {rate:.4f}", flush=True)

# ============================================================
# Save results
# ============================================================
results = {
    'method': 'frozen_clip_vit_b16_linear_probe',
    'holdout': 'sd_generated',
    'best_C': best_C,
    'val_auroc': float(best_val_auroc),
    'val_optimal_threshold': val_optimal_thresh,
    'test_prob_stats': {
        'min': float(test_probs.min()),
        'q25': float(np.percentile(test_probs, 25)),
        'median': float(np.median(test_probs)),
        'mean': float(test_probs.mean()),
        'q75': float(np.percentile(test_probs, 75)),
        'max': float(test_probs.max()),
    },
    'detection_rate_at_0.5': det_rate_05,
    'detection_rate_at_val_optimal': det_rate_youden,
    'lr_coef_norm': float(np.linalg.norm(best_lr.coef_)),
    'lr_intercept': float(best_lr.intercept_[0]),
}
with open(RESULTS_DIR / "summary.json", 'w') as f:
    json.dump(results, f, indent=2)

# Save per-image predictions
pred_df = pd.DataFrame({
    'path': test_df['path'].values,
    'label': test_labels,
    'frozen_clip_prob': test_probs,
})
pred_df.to_parquet(RESULTS_DIR / "per_image_predictions.parquet", index=False)

print(f"\n{'=' * 60}", flush=True)
print(f"COMPARISON TO FINE-TUNED ENSEMBLE", flush=True)
print(f"{'=' * 60}", flush=True)
print(f"  Fine-tuned ensemble (previous run): 0.0002", flush=True)
print(f"  Frozen CLIP linear probe @ 0.5:     {det_rate_05:.4f}", flush=True)
print(f"  Frozen CLIP linear probe @ optimal: {det_rate_youden:.4f}", flush=True)
print(f"\nResults saved to: {RESULTS_DIR}", flush=True)
print(f"Total wall time: {(time.time() - t_total) / 60:.1f} min", flush=True)

Device: NVIDIA H100 80GB HBM3
Loading master CSV...
  Train+val: 417,063
  Test (sd_generated): 10,000
  Train+val labels: {1: 213553, 0: 203510}
  Test labels: {1: 10000}

  Train: 333,651  Val: 83,412  Test: 10,000

Loading frozen CLIP ViT-B/16 (OpenAI weights)...
  CLIP feature dim: 512

FEATURE EXTRACTION

  Extracting features: TRAIN (333,651 images)
    [12,800/333,651] 2192 img/s  ETA: 146s
    [25,600/333,651] 2749 img/s  ETA: 112s
    [38,400/333,651] 3027 img/s  ETA: 98s
    [51,200/333,651] 3170 img/s  ETA: 89s
    [64,000/333,651] 3279 img/s  ETA: 82s
    [76,800/333,651] 3324 img/s  ETA: 77s
    [89,600/333,651] 3361 img/s  ETA: 73s
    [102,400/333,651] 3399 img/s  ETA: 68s
    [115,200/333,651] 3419 img/s  ETA: 64s
    [128,000/333,651] 3431 img/s  ETA: 60s
    [140,800/333,651] 3436 img/s  ETA: 56s
    [153,600/333,651] 3444 img/s  ETA: 52s
    [166,400/333,651] 3443 img/s  ETA: 49s
    [179,200/333,651] 3447 img/s  ETA: 45s
    [192,000/333,651] 3457 img/s  ETA: 41s
  